# Сверка ИНН: defaults vs ДР vs CRM

Отдельная тетрадка для проверки:
- пересечения ИНН между новой выгрузкой `dbd_def_23_25.csv` и `default_data.pkl`;
- наличия ИНН из новой выгрузки в CRM (`X_INN`);
- наличия валидных ФП/СФП по этим ИНН в CRM (`NUMBER_FP_SFP`, при наличии `TYPE_FP`).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)


def normalize_inn(s):
    """Приводит ИНН к единому строковому формату (10 или 12 знаков)."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def pick_inn_col(df: pd.DataFrame, label: str) -> str:
    cols = {c.lower().strip(): c for c in df.columns}
    for key in ["inn", "x_inn", "инн"]:
        if key in cols:
            return cols[key]
    raise KeyError(f"В источнике {label} не найдена колонка ИНН (ожидались inn/x_inn/ИНН)")


def read_csv_flexible(path: Path) -> pd.DataFrame:
    """Читает CSV с автоопределением разделителя."""
    return pd.read_csv(path, dtype=str, low_memory=False, sep=None, engine="python")


def find_file(filename: str, candidates: list[Path]) -> Path:
    for base in candidates:
        p = base / filename
        if p.exists():
            return p
    for base in candidates:
        if base.exists():
            found = list(base.rglob(filename))
            if found:
                return found[0]
    raise FileNotFoundError(f"Не найден файл: {filename}")


In [ ]:
BASE_CANDIDATES = [
    Path("/home/jovyan/documents/DMUKP_FP_DEF/data"),
    Path("e:/DMUKP/data"),
    Path("e:/DMUKP/sources"),
    Path("e:/DMUKP"),
    Path.cwd(),
]

new_defaults_path = find_file("dbd_def_23_25.csv", BASE_CANDIDATES)
dr_defaults_path = find_file("default_data.pkl", BASE_CANDIDATES)
crm_path = find_file("crm_last_version.csv", BASE_CANDIDATES)

print("Файлы для проверки:")
print(f"- новая выгрузка defaults: {new_defaults_path}")
print(f"- выгрузка ДР:             {dr_defaults_path}")
print(f"- CRM:                     {crm_path}")

new_defaults = read_csv_flexible(new_defaults_path)
new_defaults.columns = new_defaults.columns.str.strip()
new_inn_col = pick_inn_col(new_defaults, "dbd_def_23_25.csv")
new_defaults["inn"] = new_defaults[new_inn_col].apply(normalize_inn)
new_defaults_inn = set(new_defaults["inn"].dropna().unique())

dr_defaults = pd.read_pickle(dr_defaults_path)
dr_defaults = dr_defaults.astype(str).replace("nan", np.nan)
dr_defaults.columns = dr_defaults.columns.str.strip()
dr_inn_col = pick_inn_col(dr_defaults, "default_data.pkl")
dr_defaults["inn"] = dr_defaults[dr_inn_col].apply(normalize_inn)
dr_defaults_inn = set(dr_defaults["inn"].dropna().unique())

df_crm = read_csv_flexible(crm_path)
df_crm.columns = df_crm.columns.str.strip()
if "X_INN" not in df_crm.columns:
    raise KeyError("В CRM-файле отсутствует колонка X_INN")

df_crm["inn"] = df_crm["X_INN"].apply(normalize_inn)


In [ ]:
# 1) Сравнение множеств ИНН: new defaults vs DR
new_only = new_defaults_inn - dr_defaults_inn
dr_only = dr_defaults_inn - new_defaults_inn
inn_intersection = new_defaults_inn & dr_defaults_inn

base_new = len(new_defaults_inn) if len(new_defaults_inn) > 0 else 1

inn_compare_summary = pd.DataFrame([
    {"metric": "ИНН в новой выгрузке defaults", "value": len(new_defaults_inn), "share_from_new": len(new_defaults_inn) / base_new},
    {"metric": "ИНН в выгрузке ДР (default_data.pkl)", "value": len(dr_defaults_inn), "share_from_new": len(dr_defaults_inn) / base_new},
    {"metric": "ИНН в пересечении", "value": len(inn_intersection), "share_from_new": len(inn_intersection) / base_new},
    {"metric": "ИНН только в новой выгрузке", "value": len(new_only), "share_from_new": len(new_only) / base_new},
    {"metric": "ИНН только в выгрузке ДР", "value": len(dr_only), "share_from_new": len(dr_only) / base_new},
])

# 2) Проверка наличия ФП/СФП в CRM для ИНН из новой выгрузки
crm_base = df_crm.copy()
if "TYPE_FP" in crm_base.columns:
    crm_base["TYPE_FP"] = (
        crm_base["TYPE_FP"].astype(str).str.strip().str.upper().replace({"FP": "ФП", "SFP": "СФП"})
    )
    crm_fp_rows = crm_base[
        crm_base["inn"].notna()
        & crm_base["NUMBER_FP_SFP"].notna()
        & crm_base["TYPE_FP"].isin(["ФП", "СФП"])
    ].copy()
else:
    crm_fp_rows = crm_base[
        crm_base["inn"].notna()
        & crm_base["NUMBER_FP_SFP"].notna()
    ].copy()

crm_all_inn = set(crm_base["inn"].dropna().unique())
crm_fp_inn = set(crm_fp_rows["inn"].dropna().unique())

coverage = pd.DataFrame({"inn": sorted(new_defaults_inn)})
coverage["in_crm"] = coverage["inn"].isin(crm_all_inn)
coverage["has_fp_sfp_in_crm"] = coverage["inn"].isin(crm_fp_inn)
coverage["status"] = np.select(
    [
        coverage["has_fp_sfp_in_crm"],
        coverage["in_crm"] & ~coverage["has_fp_sfp_in_crm"],
    ],
    [
        "есть в CRM с ФП/СФП",
        "есть в CRM, но без валидного ФП/СФП",
    ],
    default="нет в CRM",
)

crm_coverage_summary = (
    coverage.groupby("status", as_index=False)["inn"]
    .nunique()
    .rename(columns={"inn": "cnt_inn"})
)
crm_coverage_summary["share"] = crm_coverage_summary["cnt_inn"] / base_new

print("=" * 90)
print("Сверка ИНН: новая выгрузка defaults vs выгрузка ДР (default_data.pkl)")
print("=" * 90)
display(inn_compare_summary)

print("\nПримеры ИНН только в новой выгрузке (до 20):")
display(pd.DataFrame({"inn": sorted(list(new_only))[:20]}))

print("\nПримеры ИНН только в выгрузке ДР (до 20):")
display(pd.DataFrame({"inn": sorted(list(dr_only))[:20]}))

print("\n" + "=" * 90)
print("Покрытие ИНН из новой выгрузки в CRM и наличие ФП/СФП")
print("=" * 90)
display(crm_coverage_summary.sort_values("cnt_inn", ascending=False))

print("\nПримеры ИНН: есть в CRM с ФП/СФП (до 20)")
display(coverage.loc[coverage["status"] == "есть в CRM с ФП/СФП", ["inn"]].head(20))

print("\nПримеры ИНН: есть в CRM, но без валидного ФП/СФП (до 20)")
display(coverage.loc[coverage["status"] == "есть в CRM, но без валидного ФП/СФП", ["inn"]].head(20))

print("\nПримеры ИНН: отсутствуют в CRM (до 20)")
display(coverage.loc[coverage["status"] == "нет в CRM", ["inn"]].head(20))

# Экспорт артефактов
out_dir = new_defaults_path.parent
compare_out = out_dir / "inn_compare_new_defaults_vs_dr.csv"
coverage_out = out_dir / "inn_coverage_new_defaults_vs_crm.csv"

inn_compare_export = pd.DataFrame({"inn": sorted(new_defaults_inn | dr_defaults_inn)})
inn_compare_export["in_new_defaults"] = inn_compare_export["inn"].isin(new_defaults_inn)
inn_compare_export["in_dr_defaults"] = inn_compare_export["inn"].isin(dr_defaults_inn)

inn_compare_export.to_csv(compare_out, index=False, encoding="utf-8-sig")
coverage.to_csv(coverage_out, index=False, encoding="utf-8-sig")

print(f"\nСохранены файлы:\n- {compare_out}\n- {coverage_out}")
